In [1]:
#solution to fix the missing game logs from a player due to injury/DNP/suspension/unexplained

#using sacramento kings 2025 as test case 

In [2]:
import sys
sys.path.append('..')

import pandas as pd

from nbainjuries import injury
from datetime import datetime

from nba_api.stats.static import players
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog

from scripts.data_cleaning_nba_api import clean_gamelog
from scripts.reorder_columns import reorder_columns
from scripts.data_cleaning_team_schedule import clean_team_schedule

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
from nba_api.stats.endpoints import leaguegamefinder

gamefinder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=1610612758,
    season_nullable='2025-26'
)
df_sac_schedule = gamefinder.get_data_frames()[0]
df_sac_schedule.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612758,SAC,Sacramento Kings,0022501200,2026-04-12,SAC @ POR,L,240,110,40,83,0.482,7,21,0.333,23,31,0.742,15,32,47,24,7,8,16,18,-12.0
1,22025,1610612758,SAC,Sacramento Kings,0022501184,2026-04-10,SAC vs. GSW,W,241,124,41,89,0.461,17,44,0.386,25,34,0.735,16,33,49,25,10,6,16,21,6.0
2,22025,1610612758,SAC,Sacramento Kings,0022501154,2026-04-07,SAC @ GSW,L,241,105,41,87,0.471,11,31,0.355,12,17,0.706,13,28,41,22,9,6,14,21,-5.0
3,22025,1610612758,SAC,Sacramento Kings,0022501141,2026-04-05,SAC vs. LAC,L,239,109,45,85,0.529,9,32,0.281,10,20,0.500,12,30,42,25,9,8,20,19,-29.0
4,22025,1610612758,SAC,Sacramento Kings,0022501128,2026-04-03,SAC vs. NOP,W,241,117,46,90,0.511,10,33,0.303,15,20,0.750,12,34,46,29,7,6,14,16,4.0


In [4]:
all_teams = teams.get_teams()
team_lookup = {t['abbreviation']: t['id'] for t in all_teams}

In [5]:
df_sac_schedule

#includes pre-season as well (12025) but all we want is the regular seasons (22025)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612758,SAC,Sacramento Kings,0022501200,2026-04-12,SAC @ POR,L,240,110,40,83,0.482,7,21,0.333,23,31,0.742,15,32,47,24,7,8,16,18,-12.0
1,22025,1610612758,SAC,Sacramento Kings,0022501184,2026-04-10,SAC vs. GSW,W,241,124,41,89,0.461,17,44,0.386,25,34,0.735,16,33,49,25,10,6,16,21,6.0
2,22025,1610612758,SAC,Sacramento Kings,0022501154,2026-04-07,SAC @ GSW,L,241,105,41,87,0.471,11,31,0.355,12,17,0.706,13,28,41,22,9,6,14,21,-5.0
3,22025,1610612758,SAC,Sacramento Kings,0022501141,2026-04-05,SAC vs. LAC,L,239,109,45,85,0.529,9,32,0.281,10,20,0.500,12,30,42,25,9,8,20,19,-29.0
4,22025,1610612758,SAC,Sacramento Kings,0022501128,2026-04-03,SAC vs. NOP,W,241,117,46,90,0.511,10,33,0.303,15,20,0.750,12,34,46,29,7,6,14,16,4.0
5,22025,1610612758,SAC,Sacramento Kings,0022501108,2026-04-01,SAC @ TOR,W,239,123,42,92,0.457,12,33,0.364,27,29,0.931,19,29,48,24,7,3,13,22,8.0
6,22025,1610612758,SAC,Sacramento Kings,0022501084,2026-03-29,SAC @ BKN,L,239,99,36,84,0.429,10,32,0.313,17,25,0.680,12,31,43,21,7,3,15,23,-17.0
7,22025,1610612758,SAC,Sacramento Kings,0022501078,2026-03-28,SAC @ ATL,L,239,113,45,93,0.484,12,32,0.375,11,13,0.846,16,26,42,27,7,1,15,18,-10.0
8,22025,1610612758,SAC,Sacramento Kings,0022501065,2026-03-26,SAC @ ORL,L,240,117,45,88,0.511,14,33,0.424,13,16,0.813,9,26,35,30,7,3,9,24,-4.0
9,22025,1610612758,SAC,Sacramento Kings,0022501047,2026-03-24,SAC @ CHA,L,241,90,37,91,0.407,9,28,0.321,7,11,0.636,10,25,35,27,4,3,12,15,-44.0


In [6]:
#fix to filter out pre-season

df_sac_schedule = df_sac_schedule[
    (df_sac_schedule['SEASON_ID'].astype(str).str.startswith('2')) &
    (df_sac_schedule['SEASON_ID'] == '22025')
]
df_sac_schedule

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612758,SAC,Sacramento Kings,0022501200,2026-04-12,SAC @ POR,L,240,110,40,83,0.482,7,21,0.333,23,31,0.742,15,32,47,24,7,8,16,18,-12.0
1,22025,1610612758,SAC,Sacramento Kings,0022501184,2026-04-10,SAC vs. GSW,W,241,124,41,89,0.461,17,44,0.386,25,34,0.735,16,33,49,25,10,6,16,21,6.0
2,22025,1610612758,SAC,Sacramento Kings,0022501154,2026-04-07,SAC @ GSW,L,241,105,41,87,0.471,11,31,0.355,12,17,0.706,13,28,41,22,9,6,14,21,-5.0
3,22025,1610612758,SAC,Sacramento Kings,0022501141,2026-04-05,SAC vs. LAC,L,239,109,45,85,0.529,9,32,0.281,10,20,0.500,12,30,42,25,9,8,20,19,-29.0
4,22025,1610612758,SAC,Sacramento Kings,0022501128,2026-04-03,SAC vs. NOP,W,241,117,46,90,0.511,10,33,0.303,15,20,0.750,12,34,46,29,7,6,14,16,4.0
5,22025,1610612758,SAC,Sacramento Kings,0022501108,2026-04-01,SAC @ TOR,W,239,123,42,92,0.457,12,33,0.364,27,29,0.931,19,29,48,24,7,3,13,22,8.0
6,22025,1610612758,SAC,Sacramento Kings,0022501084,2026-03-29,SAC @ BKN,L,239,99,36,84,0.429,10,32,0.313,17,25,0.680,12,31,43,21,7,3,15,23,-17.0
7,22025,1610612758,SAC,Sacramento Kings,0022501078,2026-03-28,SAC @ ATL,L,239,113,45,93,0.484,12,32,0.375,11,13,0.846,16,26,42,27,7,1,15,18,-10.0
8,22025,1610612758,SAC,Sacramento Kings,0022501065,2026-03-26,SAC @ ORL,L,240,117,45,88,0.511,14,33,0.424,13,16,0.813,9,26,35,30,7,3,9,24,-4.0
9,22025,1610612758,SAC,Sacramento Kings,0022501047,2026-03-24,SAC @ CHA,L,241,90,37,91,0.407,9,28,0.321,7,11,0.636,10,25,35,27,4,3,12,15,-44.0


In [7]:
# best modular fix: 

from nba_api.stats.endpoints import leaguegamefinder

gamefinder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=1610612758,
    season_nullable='2025-26',
    season_type_nullable='Regular Season'
)
df_sac_schedule = gamefinder.get_data_frames()[0]
df_sac_schedule.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612758,SAC,Sacramento Kings,0022501200,2026-04-12,SAC @ POR,L,240,110,40,83,0.482,7,21,0.333,23,31,0.742,15,32,47,24,7,8,16,18,-12.0
1,22025,1610612758,SAC,Sacramento Kings,0022501184,2026-04-10,SAC vs. GSW,W,241,124,41,89,0.461,17,44,0.386,25,34,0.735,16,33,49,25,10,6,16,21,6.0
2,22025,1610612758,SAC,Sacramento Kings,0022501154,2026-04-07,SAC @ GSW,L,241,105,41,87,0.471,11,31,0.355,12,17,0.706,13,28,41,22,9,6,14,21,-5.0
3,22025,1610612758,SAC,Sacramento Kings,0022501141,2026-04-05,SAC vs. LAC,L,239,109,45,85,0.529,9,32,0.281,10,20,0.500,12,30,42,25,9,8,20,19,-29.0
4,22025,1610612758,SAC,Sacramento Kings,0022501128,2026-04-03,SAC vs. NOP,W,241,117,46,90,0.511,10,33,0.303,15,20,0.750,12,34,46,29,7,6,14,16,4.0


In [8]:
print(df_sac_schedule.columns.tolist())

['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']


In [9]:
df_sac_schedule.head(1).T

,0
SEASON_ID,22025
TEAM_ID,1610612758
TEAM_ABBREVIATION,SAC
TEAM_NAME,Sacramento Kings
GAME_ID,0022501200
GAME_DATE,2026-04-12
MATCHUP,SAC @ POR
WL,L
MIN,240
PTS,110


In [10]:
# player_status (conceptual, not yet built)
# = team_schedule (every game a team played)
#  LEFT JOIN game_logs ON game_id + player_id
#  → row exists in game_logs? played
#  → row missing? check nbainjuries for that date/player → explained absence or unexplained DNP

In [11]:
raynaud = players.find_players_by_full_name("Maxime Raynaud")
raynaud_id = raynaud[0]['id']

raynaud_gamelog = playergamelog.PlayerGameLog(player_id=raynaud_id, season='2025-26')
df_raynaud = raynaud_gamelog.get_data_frames()[0]
df_raynaud.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22025,1642875,0022501200,"Apr 12, 2026",SAC @ POR,L,37,7,10,0.700,3,4,0.75,4,4,1.000,2,7,9,2,1,2,3,0,21,-9,1
1,22025,1642875,0022501184,"Apr 10, 2026",SAC vs. GSW,W,39,9,12,0.750,2,4,0.50,3,5,0.600,1,8,9,4,1,0,3,3,23,20,1
2,22025,1642875,0022501154,"Apr 07, 2026",SAC @ GSW,L,26,7,14,0.500,0,1,0.00,3,3,1.000,2,5,7,1,0,0,2,0,17,-4,1
3,22025,1642875,0022501141,"Apr 05, 2026",SAC vs. LAC,L,33,5,12,0.417,0,1,0.00,1,2,0.500,4,11,15,2,1,1,2,1,11,-11,1
4,22025,1642875,0022501128,"Apr 03, 2026",SAC vs. NOP,W,29,11,14,0.786,0,1,0.00,6,7,0.857,2,7,9,4,0,0,1,2,28,1,1


In [12]:
df_raynaud = clean_gamelog(df_raynaud, team_lookup)
df_raynaud.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,opponent_team_id,is_home,WL,MIN,FGM,FGA,FG3M,FG3A,FTM,FTA,OREB,DREB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS
0,22025,1642875,0022501200,"Apr 12, 2026",1610612757,False,L,37,7,10,3,4,4,4,2,7,2,1,2,3,0,21,-9
1,22025,1642875,0022501184,"Apr 10, 2026",1610612744,True,W,39,9,12,2,4,3,5,1,8,4,1,0,3,3,23,20
2,22025,1642875,0022501154,"Apr 07, 2026",1610612744,False,L,26,7,14,0,1,3,3,2,5,1,0,0,2,0,17,-4
3,22025,1642875,0022501141,"Apr 05, 2026",1610612746,True,L,33,5,12,0,1,1,2,4,11,2,1,1,2,1,11,-11
4,22025,1642875,0022501128,"Apr 03, 2026",1610612740,True,W,29,11,14,0,1,6,7,2,7,4,0,0,1,2,28,1


In [13]:
df_sac_schedule = clean_team_schedule(df_sac_schedule, team_lookup)

In [14]:
print(df_sac_schedule.columns.tolist())
print(df_raynaud.columns.tolist())

['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'opponent_team_id', 'is_home', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'opponent_team_id', 'is_home', 'WL', 'MIN', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS']


In [ ]:
#left joining the kings schedule with the raynaud schedule

merged = df_sac_schedule.merge(
    df_raynaud,
    left_on='GAME_ID',
    right_on='Game_ID',
    how='left',
    indicator=True
)

merged[merged['_merge'] == 'left_only']

,SEASON_ID_x,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE_x,opponent_team_id_x,is_home_x,WL_x,MIN_x,PTS_x,FGM_x,FGA_x,FG3M_x,FG3A_x,FTM_x,FTA_x,OREB_x,DREB_x,AST_x,STL_x,BLK_x,TOV_x,PF_x,PLUS_MINUS_x,SEASON_ID_y,Player_ID,Game_ID,GAME_DATE_y,opponent_team_id_y,is_home_y,WL_y,MIN_y,FGM_y,FGA_y,FG3M_y,FG3A_y,FTM_y,FTA_y,OREB_y,DREB_y,AST_y,STL_y,BLK_y,TOV_y,PF_y,PTS_y,PLUS_MINUS_y,_merge
68,22025,1610612758,SAC,Sacramento Kings,0022500234,2025-11-16,1610612759,False,L,241,110,41,87,12,33,16,22,5,32,25,10,3,12,17,-13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
69,22025,1610612758,SAC,Sacramento Kings,0022500044,2025-11-14,1610612750,False,L,240,110,41,93,12,33,16,27,13,31,32,12,4,15,26,-14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
71,22025,1610612758,SAC,Sacramento Kings,0022500212,2025-11-11,1610612743,True,L,240,108,41,85,10,33,16,23,9,26,31,4,3,10,23,-14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
75,22025,1610612758,SAC,Sacramento Kings,0022500162,2025-11-03,1610612743,False,L,240,124,47,90,9,27,21,25,8,39,27,5,1,15,24,-6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
77,22025,1610612758,SAC,Sacramento Kings,0022500132,2025-10-29,1610612741,False,L,241,113,43,82,7,28,20,26,5,26,28,10,2,10,17,-13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
78,22025,1610612758,SAC,Sacramento Kings,0022500126,2025-10-28,1610612760,False,L,240,101,36,85,10,29,19,21,9,37,21,11,3,17,16,-6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
79,22025,1610612758,SAC,Sacramento Kings,0022500113,2025-10-26,1610612747,True,L,240,120,45,101,18,43,12,18,13,27,27,13,4,15,26,-7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
80,22025,1610612758,SAC,Sacramento Kings,0022500098,2025-10-24,1610612762,True,W,240,105,40,83,14,32,11,15,5,28,23,10,4,13,18,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [17]:
print(len(gap_dates))

8


In [18]:
#verifying raynaud didn't play in the game listed in the previous cell output 

gap_dates = merged[merged['_merge'] == 'left_only']['GAME_DATE_x'].tolist()

for date_str in gap_dates:
    date_obj = datetime.strptime(date_str, '%Y-%m-%d')
    report_time = datetime(date_obj.year, date_obj.month, date_obj.day, 17, 30)
    
    try:
        report = injury.get_reportdata(report_time, return_df=True)
        match = report[report['Player Name'].str.contains('Raynaud', case=False, na=False)]
        
        if match.empty:
            print(f"{date_str}: NOT on injury report (likely coach's decision)")
        else:
            print(f"{date_str}: ON injury report — {match['Current Status'].values[0]}, {match['Reason'].values[0]}")
    except Exception as e:
        print(f"{date_str}: report unavailable — {e}")

Validated Injury-Report_2025-11-16_05PM.
2025-11-16: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-11-14_05PM.
2025-11-14: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-11-11_05PM.
2025-11-11: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-11-03_05PM.
2025-11-03: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-10-29_05PM.
2025-10-29: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-10-28_05PM.
2025-10-28: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-10-26_05PM.
2025-10-26: NOT on injury report (likely coach's decision)
Validated Injury-Report_2025-10-24_05PM.
2025-10-24: NOT on injury report (likely coach's decision)


In [20]:
#confirm start/end dates for kings regular season and raynaud regular season

df_raynaud['GAME_DATE'] = pd.to_datetime(df_raynaud['GAME_DATE'])
df_sac_schedule['GAME_DATE'] = pd.to_datetime(df_sac_schedule['GAME_DATE'])

print(df_sac_schedule['GAME_DATE'].min(), df_sac_schedule['GAME_DATE'].max())
print(df_raynaud['GAME_DATE'].min(), df_raynaud['GAME_DATE'].max())

2025-10-22 00:00:00 2026-04-12 00:00:00
2025-10-22 00:00:00 2026-04-12 00:00:00
